# 04. Decoder LoRA: attention targets, depth and rank

First real training comparison. All default experiments start independently from the local base
model with the same train/val data, seed, learning rate, epochs and image settings.
Each best checkpoint is chosen by validation loss; final generated-response comparison is in 07.
Only reviewed train pages contribute gradients. Keep test pages hidden.

Start with q/v in the last eight decoder blocks, then expand to all attention projections,
then attention plus feed-forward projections. Keep vision and projector frozen throughout.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "models.json").exists():
    raise FileNotFoundError("Start the kernel in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
print("Interpreter:", sys.executable)
print("Dataset:", H.PAGE_DIR)
print("Local base model:", H.MODEL_DIR)


In [ ]:
pages = H.read_pages()
train_pages, val_pages = H.partition(pages, "train"), H.partition(pages, "val")
EXPERIMENTS = [
    H.experiment("decoder_qv_last8_r4", decoder="qv", last_layers=8, rank=4),
    H.experiment("decoder_attention_last8_r4", decoder="attention", last_layers=8, rank=4),
    H.experiment("decoder_attention_ffn_last8_r4", decoder="attention_ffn", last_layers=8, rank=4),
]
# Run after the initial comparison, changing one factor at a time:
# H.experiment("decoder_qv_last4_r4", decoder="qv", last_layers=4, rank=4)
# H.experiment("decoder_qv_all_r4", decoder="qv", last_layers=None, rank=4)
# H.experiment("decoder_qv_last8_r8", decoder="qv", last_layers=8, rank=8)
# H.experiment("decoder_qv_last8_r4_tiles2", decoder="qv", last_layers=8, rank=4, max_tiles=2)
directories = []
for config in EXPERIMENTS:
    directory = H.train_run(config, train_pages, val_pages)
    directories.append(str(directory))
    display(json.loads((directory / "summary.json").read_text()))


Change depth only after deciding the target set; change rank only after deciding depth.
Repeat promising configurations with another seed and new run names before treating small gains
as reliable. A resolution experiment also needs its own matching frozen baseline in 07.
On OOM, restart the kernel and reduce tiles/depth; do not silently change data or settings midway.
